In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from scipy.stats import boxcox
from category_encoders import TargetEncoder


In [2]:
df = pd.read_csv("../data/income_data.csv")
df.info()

In [3]:
df.shape

In [4]:
df = df.drop_duplicates()

**Feature Engineering**

In [5]:
df['native-country'] = np.where(df['native-country'] == 'United-States', 'United-States', 'Other')
df['capital-neto'] = df['capital-gain']-df['capital-loss']

**Eliminación de Variables**

In [6]:
df = df.drop(columns=['fnlwgt','education','capital-gain','capital-loss'])

## Tratamiento de Outliers

En el EDA identificamos outliers mediante el criterio IQR en todas las variables numéricas. A continuación justificamos variable por variable por qué no se aplica eliminación de registros y qué estrategia alternativa se adopta en cada caso.

### age → Transformación Box-Cox (sin eliminación de outliers)
El análisis IQR detectó únicamente **215 registros (0.44%)** fuera de los límites. Se trata de una proporción ínfima que no justifica ninguna intervención agresiva. Además, los valores extremos de edad (personas mayores de 75 años) son perfectamente válidos y representativos de la realidad censal: existen trabajadores activos de más de 80 años en el mercado laboral estadounidense, por lo que eliminarlos introduciría un sesgo artificial. En su lugar, aplicamos la transformación Box-Cox, que corrige la ligera asimetría positiva (skewness = 0.56) observada en el EDA y reduce el peso relativo de esos valores extremos sin descartar ningún dato legítimo.

### capital-gain y capital-loss - Eliminación de columnas (sustituidas por capital-neto)
Estas dos variables presentaban los outliers más extremos del dataset: curtosis de 152.5 y 20.0 respectivamente, con más del 90% de los registros en valor cero y picos aislados de hasta 99.999 en capital-gain. Sin embargo, estos valores no son errores de medición sino rendimientos financieros reales de individuos con patrimonio: eliminar esos registros significaría borrar precisamente a los individuos más ricos del dataset, destruyendo señal predictiva clave.
La solución adoptada en el EDA fue crear la variable capital-neto = capital-gain − capital-loss, que consolida el resultado financiero neto en una sola columna y reduce drásticamente el número de variables a gestionar. Al eliminar las originales, el problema de outliers extremos en esas dos columnas queda disuelto por construcción.

### educational-num - Sin intervención
Esta variable ordinal toma valores enteros del 1 al 16, correspondientes a niveles educativos bien definidos y acotados. El EDA mostró que es la **variable más simétrica** del conjunto (skewness = 0, curtosis moderada) y no presenta outliers estadísticos significativos. Cualquier valor extremo (p.ej. nivel 1 o nivel 16) corresponde a una categoría educativa real (sin estudios / Doctorado), por lo que no existe ninguna justificación para intervenir.

### hours-per-week - Winsorización P1–P99 (sin eliminación de registros)
Esta variable concentra el caso más llamativo: el criterio IQR marcó como outliers 13.486 registros (27.64%) del dataset. Sin embargo, este porcentaje tan elevado es precisamente la señal de que el criterio IQR no es adecuado aquí: cuando casi un tercio de los datos queda fuera de los límites, el problema no son los datos sino la métrica de detección. Los valores extremos (1 hora/semana o 99 horas/semana) son perfectamente plausibles en una encuesta censal que captura trabajadores a tiempo parcial mínimo, pluriempleados y trabajadores con dedicación extrema.
Eliminar esos registros sesgaría el modelo hacia una jornada laboral estándar de 40 horas, perdiendo información valiosa sobre los extremos del mercado laboral. En su lugar, aplicamos winsorización al percentil 1–99 (límites: 8h y 80h), que recorta únicamente los valores verdaderamente anómalos en los extremos manteniendo la información de todos los registros.

**Boxcox y Winsorización**

In [ ]:
# Transformación Box-Cox sobre 'age'.
# Justificación: el EDA detectó solo 215 outliers (0.44%) en esta variable, todos ellos valores de edad perfectamente válidos (trabajadores mayores de 75 años).
# No se eliminan registros. La Box-Cox corrige la asimetría positiva moderada (skewness=0.56) reduciendo el peso relativo de los extremos sin descartar ningún dato legítimo.
df['age'], lam = boxcox(df['age'])

In [ ]:
# Winsorización de 'hours-per-week' al percentil 1-99.
# Justificación: el EDA marcó 13.486 registros (27.64%) como outliers por IQR, un porcentaje tan elevado que evidencia que el criterio IQR no es adecuado aquí.
# Los valores extremos (p.ej. 1h o 99h semanales) son horas reales de trabajadores a tiempo parcial mínimo o con dedicación muy alta; eliminarlos sesgaría el modelo.
# La winsorización recorta únicamente los valores en los extremos absolutos (P1=8h, P99=80h) manteniendo todos los registros y preservando la señal predictiva de los extremos reales.

# Calculamos los percentiles 1 y 99 para hours-per-week
lower_hours = df['hours-per-week'].quantile(0.01)
upper_hours = df['hours-per-week'].quantile(0.99)

print(f"Límite inferior para el recorte (P1): {lower_hours} horas")
print(f"Límite superior para el recorte (P99): {upper_hours} horas\n")

# Aplicamos la winsorización simétrica usando .clip()
# Guardamos el resultado en una nueva columna para mantener la original intacta
df['hours-per-week'] = df['hours-per-week'].clip(lower=lower_hours, upper=upper_hours)

**División en Train y Test**

In [9]:
X = df.drop(columns=['income'])
y = df['income'].map({'<=50K': 0, '>50K': 1})

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Verificamos las dimensiones de los nuevos conjuntos
print(f"Dimensiones de X_train: {X_train.shape}")
print(f"Dimensiones de X_test:  {X_test.shape}")
print("\nDistribución del target en el conjunto de entrenamiento:")
print(y_train.value_counts(normalize=True))

**Encoding**

In [11]:
categoric_cols = X_train.select_dtypes(include ='object').columns.tolist()
categoric_cols

**One-Hot-Encoding**: Se aplicó One-Hot Encoding en marital-status, relationship, native-country, gender y race porque son variables nominales con baja cardinalidad y sin un orden lógico interno. Al transformarlas en columnas binarias independientes (de ceros y unos), el modelo puede interpretar su impacto de forma limpia y directa sin el riesgo de asumir jerarquías artificiales entre categorías como el género o el estado civil.

In [12]:
baja_cardinalidad = ['marital-status', 'relationship', 'native-country', 'gender', 'race']

X_train = pd.get_dummies(X_train, columns=baja_cardinalidad, drop_first=True, dtype=int)
X_test = pd.get_dummies(X_test, columns=baja_cardinalidad, drop_first=True, dtype=int)

**Target Encoding**: Se aplicó Target Encoding en occupation y workclass debido a su alta cardinalidad (15 y 7 categorías respectivamente), evitando así multiplicar innecesariamente las columnas del dataset si hubiéramos usado One-Hot Encoding. En su lugar, ambas variables se reducen a una única columna numérica que reemplaza cada texto por su probabilidad real de ganar más de 50K, simplificando la estructura de los datos y maximizando la señal predictiva del modelo.

In [ ]:
# MÉTODO 1: Target Encoding (Para variables con muchas categorías / Alta Cardinalidad)
# Reemplaza 'occupation' y 'workclass' por la media del target en Train
alta_cardinalidad = ['occupation', 'workclass']

encoder_target = TargetEncoder(cols=alta_cardinalidad)
# El fit se hace SOLO en Train, y se aplica (transform) en ambos
X_train = encoder_target.fit_transform(X_train, y_train)
X_test = encoder_target.transform(X_test)

Comprobación de que tienen las mismas columnas

In [ ]:
# Alineación final (Garantiza que Train y Test tengan las mismas columnas)
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

print(f"Dimensiones finales de X_train: {X_train.shape}")
print("Columnas listas para el modelo:")
print(X_train.columns.tolist())

In [ ]:
# Concatenamos X e y para Train y Test respectivamente
df_train_final = pd.concat([X_train, y_train], axis=1)
df_test_final = pd.concat([X_test, y_test], axis=1)

# Guardamos en la carpeta de datos (asegúrate de que la ruta exista)
# Usamos index=False para que no te cree una columna extra e incómoda con los índices antiguos
df_train_final.to_csv("../data/train.csv", index=False)
df_test_final.to_csv("../data/test.csv", index=False)

print("¡Archivos guardados con éxito en la carpeta '../data/'!")
print(f"Registros en Train: {df_train_final.shape[0]} | Columnas: {df_train_final.shape[1]}")
print(f"Registros en Test:  {df_test_final.shape[0]} | Columnas: {df_test_final.shape[1]}")